# Model Weight Rotation Lab

This notebook creates and tests a dynamic weight rotation system that can switch between different model configurations based on context.

In [ ]:
# CELL 1: Environment Setup

import os
import json
import torch
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from enum import Enum

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# CELL 2: Define Rotation Configuration

class DomainType(Enum):
    MATH = "math"
    STRUCTURED = "structured"
    INSTRUCTION = "instruction"
    FACT = "fact"
    GENERAL = "general"

@dataclass
class WeightVariant:
    name: str
    quant_type: str
    domain_specialization: DomainType
    temperature: float
    top_p: float
    top_k: int
    system_prompt: Optional[str] = None

# Define weight variants for rotation
WEIGHT_VARIANTS = [
    WeightVariant(
        name="base_q4",
        quant_type="Q4_K_M",
        domain_specialization=DomainType.GENERAL,
        temperature=0.7,
        top_p=0.9,
        top_k=40,
        system_prompt="You are a helpful assistant."
    ),
    WeightVariant(
        name="math_specialist",
        quant_type="Q4_K_M",
        domain_specialization=DomainType.MATH,
        temperature=0.3,
        top_p=0.8,
        top_k=20,
        system_prompt="You are a mathematical reasoning expert. Show your work step by step."
    ),
    WeightVariant(
        name="structured_specialist",
        quant_type="Q4_K_M",
        domain_specialization=DomainType.STRUCTURED,
        temperature=0.2,
        top_p=0.95,
        top_k=10,
        system_prompt="You are a structured data expert. Output must be valid JSON or follow exact formatting."
    ),
    WeightVariant(
        name="instruction_specialist",
        quant_type="Q4_K_M",
        domain_specialization=DomainType.INSTRUCTION,
        temperature=0.6,
        top_p=0.85,
        top_k=30,
        system_prompt="You are an instruction-following expert. Follow the exact instructions given."
    ),
]

print(f"Defined {len(WEIGHT_VARIANTS)} weight variants for rotation:")
for variant in WEIGHT_VARIANTS:
    print(f"  - {variant.name}: {variant.domain_specialization.value} (temp={variant.temperature})")

In [ ]:
# CELL 3: Domain Classifier

class DomainClassifier:
    """Classifies input prompts to determine domain type for weight selection."""
    
    def __init__(self):
        self.math_keywords = [
            'calculate', 'solve', 'compute', 'equation', 'formula',
            'percent', 'ratio', 'fraction', 'multiply', 'divide',
            'add', 'subtract', 'sum', 'total', 'average', 'number'
        ]
        self.structured_keywords = [
            'json', 'format', 'output', 'structure', 'array', 'object',
            'schema', 'list', 'dictionary', 'key', 'value', 'parse'
        ]
        self.instruction_keywords = [
            'follow', 'instruction', 'step', 'procedure', 'do this',
            'complete', 'execute', 'perform', 'task', 'command'
        ]
        self.fact_keywords = [
            'what is', 'who is', 'when was', 'where is', 'tell me about',
            'explain', 'describe', 'define', 'history', 'information'
        ]
    
    def classify(self, prompt: str) -> DomainType:
        """Classify prompt into domain type."""
        prompt_lower = prompt.lower()
        
        scores = {
            DomainType.MATH: sum(1 for kw in self.math_keywords if kw in prompt_lower),
            DomainType.STRUCTURED: sum(1 for kw in self.structured_keywords if kw in prompt_lower),
            DomainType.INSTRUCTION: sum(1 for kw in self.instruction_keywords if kw in prompt_lower),
            DomainType.FACT: sum(1 for kw in self.fact_keywords if kw in prompt_lower),
        }
        
        # Return domain with highest score, or GENERAL if no match
        max_score = max(scores.values())
        if max_score == 0:
            return DomainType.GENERAL
        
        return max(scores, key=scores.get)

classifier = DomainClassifier()

# Test classifier
test_prompts = [
    "Calculate 15% of 240",
    "Output the result as JSON",
    "Follow these steps to install Python",
    "What is the capital of France?",
    "Tell me a story"
]

print("Domain classifier test:")
for prompt in test_prompts:
    domain = classifier.classify(prompt)
    print(f"  '{prompt}' -> {domain.value}")

In [ ]:
# CELL 4: Weight Rotation Manager

class WeightRotationManager:
    """Manages dynamic weight rotation based on domain classification."""
    
    def __init__(self, variants: List[WeightVariant]):
        self.variants = variants
        self.classifier = DomainClassifier()
        self.current_variant = None
        self.rotation_history = []
        self.domain_counts = {domain: 0 for domain in DomainType}
    
    def select_variant(self, prompt: str) -> WeightVariant:
        """Select appropriate weight variant based on prompt domain."""
        domain = self.classifier.classify(prompt)
        
        # Find variant with matching domain specialization
        for variant in self.variants:
            if variant.domain_specialization == domain:
                self.current_variant = variant
                self.rotation_history.append({
                    'timestamp': len(self.rotation_history),
                    'domain': domain.value,
                    'variant': variant.name
                })
                self.domain_counts[domain] += 1
                return variant
        
        # Fallback to general
        for variant in self.variants:
            if variant.domain_specialization == DomainType.GENERAL:
                self.current_variant = variant
                return variant
    
    def get_rotation_stats(self) -> Dict:
        """Get statistics about rotation usage."""
        return {
            'total_rotations': len(self.rotation_history),
            'domain_distribution': self.domain_counts,
            'current_variant': self.current_variant.name if self.current_variant else None
        }

rotation_manager = WeightRotationManager(WEIGHT_VARIANTS)
print("Weight rotation manager initialized")
print(f"Loaded {len(WEIGHT_VARIANTS)} variants")

In [ ]:
# CELL 5: LLaMA.cpp Interface for Rotation

import subprocess
import json

class LlamaCppRotator:
    """Interface to llama.cpp with dynamic parameter rotation."""
    
    def __init__(self, model_path: str):
        self.model_path = model_path
        self.rotation_manager = WeightRotationManager(WEIGHT_VARIANTS)
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        """Generate response with automatic weight rotation."""
        # Select variant based on domain
        variant = self.rotation_manager.select_variant(prompt)
        
        # Build command with variant-specific parameters
        cmd = [
            './llama-cli',
            '-m', self.model_path,
            '-p', prompt,
            '-n', str(max_tokens),
            '--temp', str(variant.temperature),
            '--top-p', str(variant.top_p),
            '--top-k', str(variant.top_k),
        ]
        
        # Add system prompt if specified
        if variant.system_prompt:
            full_prompt = f"{variant.system_prompt}\n\n{prompt}"
            cmd.extend(['-p', full_prompt])
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            return result.stdout
        except subprocess.TimeoutExpired:
            return "[TIMEOUT] Generation took too long"
        except Exception as e:
            return f"[ERROR] {str(e)}"
    
    def get_stats(self) -> Dict:
        return self.rotation_manager.get_rotation_stats()

print("LLaMA.cpp rotator interface defined")

In [ ]:
# CELL 6: Download Model for Rotation Testing

from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_PATH = "/tmp/modelscope_cache/Qwen/Qwen2.5-1.5B-Instruct"

print(f"Downloading {MODEL_ID}...")
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_PATH,
    local_dir_use_symlinks=False
)
print(f"Model downloaded to {MODEL_PATH}")

In [ ]:
# CELL 7: Clone and Build llama.cpp

import os

LLAMA_CPP_PATH = "/tmp/llama.cpp"

if not os.path.exists(LLAMA_CPP_PATH):
    print("Cloning llama.cpp...")
    !git clone https://github.com/ggerganov/llama.cpp.git {LLAMA_CPP_PATH}
else:
    print("llama.cpp already cloned")

print("Building llama.cpp...")
!cd {LLAMA_CPP_PATH} && make -j$(nproc)
print("llama.cpp built successfully")

In [ ]:
# CELL 8: Convert to GGUF with Multiple Quantizations

import os

GGUF_OUTPUT_DIR = "/tmp/gguf_variants"
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

# Define quantization types for rotation
QUANT_TYPES = ['Q4_K_M', 'Q5_K_M', 'Q6_K']

for quant in QUANT_TYPES:
    print(f"Converting to {quant}...")
    !cd {LLAMA_CPP_PATH} && ./convert-hf-to-gguf.py {MODEL_PATH} --outfile {GGUF_OUTPUT_DIR}/qwen25-15b-{quant.lower()}.gguf --quantize {quant}
    print(f"  -> {GGUF_OUTPUT_DIR}/qwen25-15b-{quant.lower()}.gguf")

print(f"\nCreated {len(QUANT_TYPES)} GGUF variants in {GGUF_OUTPUT_DIR}")

In [ ]:
# CELL 9: Rotation Benchmark Dataset

rotation_benchmark = [
    {
        "id": "rot-001",
        "domain": "math",
        "prompt": "A train travels 120 miles in 2 hours. What is its average speed in miles per hour?",
        "expected_type": "math_specialist"
    },
    {
        "id": "rot-002",
        "domain": "structured",
        "prompt": "Extract the name and age from this text and output as JSON: 'John is 25 years old'",
        "expected_type": "structured_specialist"
    },
    {
        "id": "rot-003",
        "domain": "instruction",
        "prompt": "Follow these steps: 1) Create a file, 2) Write 'Hello', 3) Save the file",
        "expected_type": "instruction_specialist"
    },
    {
        "id": "rot-004",
        "domain": "fact",
        "prompt": "What is the largest planet in our solar system?",
        "expected_type": "base_q4"
    },
    {
        "id": "rot-005",
        "domain": "math",
        "prompt": "If a shirt costs $20 and is 25% off, what is the final price?",
        "expected_type": "math_specialist"
    },
]

print(f"Rotation benchmark: {len(rotation_benchmark)} test cases")
for item in rotation_benchmark:
    print(f"  {item['id']}: {item['domain']} -> {item['expected_type']}")

In [ ]:
# CELL 10: Run Rotation Test

import json
from pathlib import Path

# Use Q4_K_M for testing (smallest, fastest)
test_model = f"{GGUF_OUTPUT_DIR}/qwen25-15b-q4_k_m.gguf"

if not Path(test_model).exists():
    print(f"Model not found: {test_model}")
    print("Available models:")
    !ls -lh {GGUF_OUTPUT_DIR}
else:
    rotator = LlamaCppRotator(test_model)
    
    results = []
    for item in rotation_benchmark:
        print(f"\nTesting {item['id']} ({item['domain']})...")
        response = rotator.generate(item['prompt'], max_tokens=256)
        
        result = {
            'id': item['id'],
            'domain': item['domain'],
            'expected_variant': item['expected_type'],
            'selected_variant': rotator.current_variant.name,
            'match': rotator.current_variant.name == item['expected_type'],
            'response': response[:200] + '...' if len(response) > 200 else response
        }
        results.append(result)
        print(f"  Expected: {item['expected_type']}")
        print(f"  Selected: {rotator.current_variant.name}")
        print(f"  Match: {result['match']}")
    
    # Save results
    rotation_results = {
        'timestamp': json.dumps({'rotation_stats': rotator.get_stats()}),
        'results': results
    }
    
    with open('/tmp/rotation_test_results.json', 'w') as f:
        json.dump(rotation_results, f, indent=2)
    
    print("\n=== Rotation Test Summary ===")
    print(f"Total tests: {len(results)}")
    matches = sum(1 for r in results if r['match'])
    print(f"Correct rotations: {matches}/{len(results)} ({matches/len(results)*100:.1f}%)")
    print(f"\nRotation stats: {json.dumps(rotator.get_stats(), indent=2)}")

In [ ]:
# CELL 11: Generate Rotation Manifest

manifest = {
    "rotation_system": "dynamic_weight_rotation",
    "model_id": MODEL_ID,
    "variants": [
        {
            "name": v.name,
            "quantization": v.quant_type,
            "domain": v.domain_specialization.value,
            "temperature": v.temperature,
            "top_p": v.top_p,
            "top_k": v.top_k,
            "system_prompt": v.system_prompt
        }
        for v in WEIGHT_VARIANTS
    ],
    "quantization_types": QUANT_TYPES,
    "gguf_output_dir": GGUF_OUTPUT_DIR,
    "rotation_test_results": "/tmp/rotation_test_results.json"
}

with open(f"{GGUF_OUTPUT_DIR}/rotation_manifest.json", 'w') as f:
    json.dump(manifest, f, indent=2)

print("Rotation manifest created")
print(json.dumps(manifest, indent=2))